# EF English Proficiency and ESL Exposure in Europe

This notebook refactors the current analysis to a cleaner, presentation-ready workflow. The goal is to preserve the existing comparison between Eurostat ESL learning indicators and EF EPI outcomes, while removing duplicated preprocessing, debug code, and statistical inconsistency.

In [1]:
import sys
import pathlib

# Ensure the project root is on the notebook path
project_root = pathlib.Path().resolve().parent
sys.path.append(str(project_root))

import eurostat
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

from src.preprocessing import load_ef_epi, clean_eurostat, merge_datasets

## Data loading

Load the Eurostat indicator and the EF EPI dataset. This section preserves the current data sources and keeps the workflow transparent.

In [2]:
datasets = ["educ_uoe_lang01"]
raw_data = {}
for ds in datasets:
    print(f"Loading Eurostat dataset: {ds}")
    raw_data[ds] = eurostat.get_data_df(ds, flags=False)


ef_path = project_root / "data" / "raw" / "efiepi_rankings.csv"
ef = load_ef_epi(str(ef_path))

print("EF dataset shape:", ef.shape)
print("Eurostat dataset shape:", raw_data['educ_uoe_lang01'].shape)

Loading Eurostat dataset: educ_uoe_lang01
Country not found: Turkey
Country not found: Turkey
Country not found: Turkey
Country not found: Turkey
Country not found: Turkey
Country not found: Turkey
Country not found: Turkey
Country not found: Turkey
EF dataset shape: (238, 7)
Eurostat dataset shape: (12133, 18)


## Preprocessing

This section applies a single, consistent Eurostat cleaning pipeline and merges EF data by ISO code and year. The comparison metric is percentile-based to avoid mixing ranks and z-scores.

In [3]:
euro_df = clean_eurostat(raw_data["educ_uoe_lang01"])
merged = merge_datasets(euro_df, ef)

merged = merged.dropna(subset=["ef_percentile", "learning_percentile"])
merged = merged.sort_values(["year", "iso3"])
latest_year = int(merged["year"].max())
merged_latest = merged[merged["year"] == latest_year]

print("Merged dataset shape:", merged.shape)
print("Latest year used for summary plots:", latest_year)
print(merged[["iso3", "year", "learning", "learning_percentile", "ef_percentile", "gap_pct"]].head())

Merged dataset shape: (165, 7)
Latest year used for summary plots: 2020
   iso3  year  learning  learning_percentile  ef_percentile   gap_pct
2   AUT  2013     99.54             0.900000       0.730769 -0.169231
18  BEL  2013     56.14             0.066667       0.653846  0.587179
30  BGR  2013     83.66             0.366667       0.346154 -0.020513
54  CZE  2013     87.00             0.500000       0.423077 -0.076923
66  DEU  2013     69.04             0.266667       0.769231  0.502564


## Exploratory analysis

The dataset now includes:

- `learning`: Eurostat English learning exposure
- `learning_percentile`: year-wise percentile rank of the ESL measure
- `ef_percentile`: year-wise percentile rank of EF proficiency
- `gap_pct`: the percentile gap between EF proficiency and ESL exposure

In [4]:
summary = merged.groupby("year")[['learning', 'learning_percentile', 'ef_percentile', 'gap_pct']].median()
print(summary)

      learning  learning_percentile  ef_percentile   gap_pct
year                                                        
2013     85.71             0.450000       0.596154  0.015385
2014     85.15             0.482759       0.645161  0.097330
2015     90.00             0.517241       0.666667  0.021839
2016     85.07             0.465517       0.637931  0.068966
2017     88.28             0.450000       0.661290  0.076344
2018     92.09             0.500000       0.650000  0.124713
2019     85.04             0.431034       0.661290  0.163515
2020     85.63             0.383333       0.650000  0.216667


## Europe choropleth: latest-year gap

This static map shows the most recent year’s relative gap between EF proficiency percentiles and ESL exposure percentiles across Europe.

In [10]:
# =========================
# STATIC MAP — latest year
# =========================

fig = px.choropleth(
    merged_latest,
    locations="iso3",
    color="gap_pct",
    hover_name="geo",
    projection="mercator",
    color_continuous_scale="RdYlGn",
    range_color=[-1, 1],
    title=f"EF vs ESL gap across Europe ({latest_year})"
)

fig.update_geos(
    scope="europe",
    showcountries=True,
    showcoastlines=True,
    coastlinecolor="gray",
    showland=True,
    landcolor="rgb(240,240,240)",
    showframe=False
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    width=950,
    height=650,
    margin=dict(l=10, r=10, t=50, b=10),
    coloraxis_colorbar_title="Gap"
)

fig.show()


In [11]:
# =========================
# ANIMATED MAP — evolution
# =========================

interactive_df = merged.dropna(subset=["gap_pct"]).copy()

fig = px.choropleth(
    interactive_df,
    locations="iso3",
    color="gap_pct",
    hover_name="geo",
    animation_frame="year",
    projection="mercator",
    color_continuous_scale="RdYlGn",
    range_color=[-1, 1],
    title="Evolution of EF vs ESL gap across Europe"
)

fig.update_geos(
    scope="europe",
    showcountries=True,
    showcoastlines=True,
    coastlinecolor="gray",
    showland=True,
    landcolor="rgb(240,240,240)",
    showframe=False
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    width=950,
    height=650,
    margin=dict(l=10, r=10, t=50, b=10),
    coloraxis_colorbar_title="Gap"
)

fig.show()

## Italy vs Europe average trajectory

This figure compares Italy’s annual percentile trajectory with the European average for the same measures.


In [7]:
italy_traj = (
    merged[merged["iso3"] == "ITA"]
    .groupby("year", as_index=False)["gap_pct"]
    .mean()
)

europe_avg = (
    merged
    .groupby("year", as_index=False)["gap_pct"]
    .mean()
    .rename(columns={"gap_pct": "europe_gap_pct"})
)

traj = pd.merge(italy_traj, europe_avg, on="year")

fig = go.Figure()

# Europe reference line
fig.add_trace(
    go.Scatter(
        x=traj["year"],
        y=traj["europe_gap_pct"],
        mode="lines",
        name="Europe average",
        line=dict(color="gray", width=2, dash="dash")
    )
)

# Italy focus line
fig.add_trace(
    go.Scatter(
        x=traj["year"],
        y=traj["gap_pct"],
        mode="lines+markers",
        name="Italy",
        line=dict(color="#d62728", width=4),
        marker=dict(size=8)
    )
)

# Baseline
fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="black"
)

fig.update_layout(
    template="plotly_white",
    title="Italy consistently underperforms relative to ESL exposure",
    title_x=0.5,
    xaxis_title="Year",
    yaxis_title="Relative performance gap",
    width=950,
    height=550,
    margin=dict(l=40, r=40, t=60, b=40),
    legend_title=""
)

fig.show()


## Interpretation and limitations

The final analysis uses percentile ranks for both ESL exposure and EF proficiency. This makes the gap metric coherent and easier to interpret. The current work is intentionally descriptive and avoids complex lag modelling or causal claims.

Limitations:

- The gap metric is relative within each year, not an absolute measure.
- Country coverage depends on ISO harmonization and available Eurostat/EPI observations.
- A full dashboard can be added later once the cleaned analytical workflow is stable.